In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
import numpy as np
import joblib

# ------------------------------------
# 1. 데이터 로드
# ------------------------------------
df = pd.read_csv(
    "/Users/mac/Desktop/project/company_data/third_week/data_csv/전국일반음식점.csv",
    encoding="CP949",
    low_memory=False
)

# ------------------------------------
# 2. 날짜 처리
# ------------------------------------
df["인허가일자"] = pd.to_datetime(df["인허가일자"], errors="coerce")
df["폐업일자"] = pd.to_datetime(df["폐업일자"], errors="coerce")

# 최근 5년만 유지
df = df[df["인허가일자"] >= "2019-01-01"].copy()

# ------------------------------------
# 3. 폐업 여부 생성
# ------------------------------------
df["폐업여부"] = df["폐업일자"].notnull().astype(int)

# ------------------------------------
# 4. 연도 생성
# ------------------------------------
df["year"] = df["인허가일자"].dt.year

# ------------------------------------
# 5. 업태별 연도별 성장(Net Growth)
#    ※ 이 데이터에서는 업태 컬럼이 보통 '업태구분명'이므로 그 기준으로 사용
# ------------------------------------
annual = (
    df.groupby(["업태구분명", "year"])
      .agg(
          신규=("인허가일자", "count"),
          폐업=("폐업여부", "sum")
      )
      .reset_index()
)
annual["net_growth"] = annual["신규"] - annual["폐업"]

df = df.merge(
    annual[["업태구분명", "year", "net_growth"]],
    on=["업태구분명", "year"],
    how="left"
)

# ------------------------------------
# 6. 영업기간 계산
# ------------------------------------
today = pd.Timestamp("2025-01-01")
df["영업종료일"] = df["폐업일자"].fillna(today)
df["영업기간"] = (df["영업종료일"] - df["인허가일자"]).dt.days.clip(lower=0)

# ------------------------------------
# 7. 면적 처리
# ------------------------------------
df["소재지면적"] = pd.to_numeric(df["소재지면적"], errors="coerce")
df["소재지면적"] = df["소재지면적"].fillna(df["소재지면적"].median())
df["log_면적"] = np.log1p(df["소재지면적"])

# ------------------------------------
# 8. 지역(시군구) 생성
# ------------------------------------
def extract_region(addr):
    try:
        parts = addr.split()
        if len(parts) >= 2:
            return parts[0] + " " + parts[1]
        return np.nan
    except Exception:
        return np.nan

df["지역"] = df["소재지전체주소"].astype(str).apply(extract_region)

# ------------------------------------
# 8-1. 지역별 폐업률 계산 (원본 '지역' 컬럼이 살아 있을 때 수행)
# ------------------------------------
지역별폐업률 = (
    df.groupby("지역")["폐업여부"]
      .mean()
      .rename("지역폐업률")
      .reset_index()
)
df = df.merge(지역별폐업률, on="지역", how="left")

# ------------------------------------
# 9. 더미 변수 생성 (지역, 업태)
# ------------------------------------
df = pd.get_dummies(df, columns=["지역"], drop_first=True)
df = pd.get_dummies(df, columns=["업태구분명"], drop_first=True)

# ------------------------------------
# 10. X, y 구성
# ------------------------------------
업태_cols = [c for c in df.columns if c.startswith("업태구분명_")]
지역_cols = [c for c in df.columns if c.startswith("지역_")]

feature_cols = ["영업기간", "net_growth", "log_면적", "지역폐업률"] + 업태_cols + 지역_cols

X = df[feature_cols].fillna(0)
y = df["폐업여부"]

# ------------------------------------
# 11. Train/Test 분리
# ------------------------------------
X_train, X_test, Y_train, Y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ------------------------------------
# 12. 모델 학습 (Logistic Regression)
# ------------------------------------
model = LogisticRegression(max_iter=10000)
model.fit(X_train, Y_train)

# ------------------------------------
# 13. 평가
# ------------------------------------
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(Y_test, pred))
print("ROC-AUC  :", roc_auc_score(Y_test, prob))
print("\nConfusion Matrix:\n", confusion_matrix(Y_test, pred))
print("\nClassification Report:\n", classification_report(Y_test, pred))

# 베이스라인 성능 따로 저장
baseline_acc = accuracy_score(Y_test, pred)
baseline_auc = roc_auc_score(Y_test, prob)

print("\nBaseline ACC :", baseline_acc)
print("Baseline AUC :", baseline_auc)

# ------------------------------------
# 

#### 베이스라인 모델 성능 저장 

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score

In [ ]:
baseline_acc = accuracy_score(Y_test, pred)
baseline_auc = roc_auc_score(Y_test, prob)

print("Baseline ACC :", baseline_acc)
print("Baseline AUC :", baseline_auc)

### 특성 엔지니어링 추가

#### 영업기간을 구간화 (비선형 효과 반영)

In [ ]:
df["영업기간_cat"] = pd.qcut(df["영업기간"], q=4, labels=False)

#### 면적 X 지역 조합 (상호작용 특성)

In [ ]:
df["면적_지역_상호작용"] = df["log_면적"] * df["net_growth"]

#### 지역별 폐업률 (시군구 위험도)

In [ ]:
def extract_region(addr):
    try:
        parts = addr.split()
        if len(parts) >= 2:
            return parts[0] + " " + parts[1]
        return np.nan
    except:
        return np.nan

df["지역"] = df["소재지전체주소"].astype(str).apply(extract_region)

In [ ]:
df["지역"] = df["소재지전체주소"].astype(str).apply(extract_region)

In [ ]:
지역별폐업률 = df.groupby("지역")["폐업여부"].mean().rename("지역폐업률")
df = df.merge(지역별폐업률, on="지역", how="left")

In [ ]:
df = pd.get_dummies(df, columns=["지역"], drop_first=True)

In [ ]:
def extract_region(addr):
    try:
        parts = addr.split()
        if len(parts) >= 2:
            return parts[0] + " " + parts[1]
        return np.nan
    except:
        return np.nan

df["지역"] = df["소재지전체주소"].astype(str).apply(extract_region)

지역별폐업률 = df.groupby("지역")["폐업여부"].mean().rename("지역폐업률")
df = df.merge(지역별폐업률, on="지역", how="left")

In [ ]:
# 지역폐업률 컬럼이 이미 있으면 삭제
if "지역폐업률" in df.columns:
    df = df.drop(columns=["지역폐업률"])

# 지역별 폐업률 다시 계산 후 merge
지역별폐업률 = (
    df.groupby("지역")["폐업여부"]
      .mean()
      .rename("지역폐업률")
      .reset_index()
)

df = df.merge(지역별폐업률, on="지역", how="left")

In [ ]:
# 1) 기존에 남아 있을 수 있는 모든 지역폐업률 관련 컬럼 제거
remove_cols = [c for c in df.columns if "지역폐업률" in c]
df = df.drop(columns=remove_cols, errors="ignore")

# 2) 지역별 폐업률 새로 계산
지역별폐업률 = (
    df.groupby("지역")["폐업여부"]
      .mean()
      .rename("지역폐업률")
      .reset_index()
)

# 3) 완전 새로 merge
df = df.merge(지역별폐업률, on="지역", how="left")

In [ ]:
 지역별폐업률 = df.groupby("지역")["폐업여부"].mean().rename("지역폐업률")
df = df.merge(지역별폐업률, on="지역", how="left")

### Random Forest 모델 변경 

##### (로지스틱 회귀는 선형 모델이기에 한계가 있음.) -> 트리 기반 모델은 비선형 관계를 자동으로 잡아줘서 성능이 더 올라감 

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=20,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, Y_train)

pred_rf = rf.predict(X_test)
prob_rf = rf.predict_proba(X_test)[:, 1]

print("RF ACC :", accuracy_score(Y_test, pred_rf))
print("RF AUC :", roc_auc_score(Y_test, prob_rf))

### XGBoost 모델 변경 

#### Grandient Boosting 모델의 업그레이드 버전으로 여러 결정 트리를 순차적으로 쌓아서 만드는 모델이다. 이전 트리가 틀린 부분을 다음 트리가 보완해서 계속하여 개선을 한다. 따라서 성능이 매우 높고 속도가 빠르며 과적합 방지 기능이 좋다!!

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="auc",
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, Y_train)

pred_xgb = xgb.predict(X_test)
prob_xgb = xgb.predict_proba(X_test)[:, 1]

print("XGB ACC :", accuracy_score(Y_test, pred_xgb))
print("XGB AUC :", roc_auc_score(Y_test, prob_xgb))

### 하이퍼파라미터 튜닝 

#### 성능에 영향을 주는 설정값을 하이퍼파라미터라고 부른다!! 트리 깊이, 트리 개수, 학습률, 샘플 사용비율, 가지치기 수치가 있는데 이 값들을 바꾸면 정확도, AUC 가 크게 변한다. 즉 하이퍼파라미터 튜닝은 모델 성능이 가장 잘 나오도록 설정값을 조절하는 과정이다!!!

##### 머신 러닝 모델의 옵셜을 최적 조합으로 자동 및 반 자동으로 찾는 과정이라고 보면 된다고 함!! (전략최적화?!)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "max_depth": [4, 6],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [200, 400],
    "subsample": [0.8, 1.0]
}

grid = GridSearchCV(
    XGBClassifier(eval_metric="auc", n_jobs=-1),
    param_grid,
    scoring="roc_auc",
    cv=3,
    verbose=1
)

grid.fit(X_train, Y_train)

print("Best Params:", grid.best_params_)
print("Best AUC:", grid.best_score_)

### 최종 모델 선정 기준 

In [ ]:
results = pd.DataFrame([
    ["Baseline(LogReg)", baseline_acc, baseline_auc],
    ["RandomForest", accuracy_score(Y_test, pred_rf), roc_auc_score(Y_test, prob_rf)],
    ["XGBoost", accuracy_score(Y_test, pred_xgb), roc_auc_score(Y_test, prob_xgb)]
], columns=["Model", "ACC", "AUC"])

print(results)

### 최종 모델 저장 

In [ ]:
import joblib
joblib.dump(xgb, "final_model.pkl")